Primary figures from just-think experiment (with some comparison against play outcomes).

In [ ]:

import json
import os
import random
import importlib
from collections import defaultdict
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
from tqdm import tqdm

import analysis_utils
from analysis_utils import tidy_game_code
import constants


main_dir = constants.PROCESSED_RES_DIR
main_save = constants.FINAL_FIGURES_DIR
fig_data_file_pth = constants.FIG_DATA_DIR


import matplotlib as mpl 
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:

base_game_objs, idx2game, game2idx, game_stimuli = (
    analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
)
human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
all_game_types, game2game_type = analysis_utils.get_game_categories(human_df)


In [ ]:

model2name = dict(constants.PAPER_MODEL2NAME)
agent2color = dict(constants.PAPER_AGENT2COLOR)
game_type2fmt = constants.PAPER_GAME_TYPE2FMT
order_game_types = list(constants.PAPER_ORDER_GAME_TYPES)
model2color = {}


In [ ]:
'''
Think data
''' 

with open(f"{main_dir}think/human_processed.json") as f:  
      h = json.load(f)                                                                          
with open(f"{main_dir}think/model_payoff.json") as f:     
      model_payoff = json.load(f)                      
think_data = {
      "human_payoff": h["human_payoff"],
      "human_fun":    h["human_fun"],
      "model_payoff": model_payoff,
}
with open(f"{main_dir}think/llm_funness.json") as f:
      model_funness = json.load(f)   

dist_df = pd.read_csv(f"{main_dir}think/payoff_cat_dists.csv")


In [ ]:


funness_df = pd.read_csv("final_processed_res/think/K_variation_funness_predictions.csv")

vary_K = np.arange(2, 11, 1)
n_sim_participants = 20
k = 6
n_bootstrap = 1000
model_src = 'ours_full'

fig, axes = plt.subplots(1, 2, figsize=(10, 4))


ax = axes[0]

agg_vals = []
h_agg_vals = []
model_ci_lower = []
model_ci_upper = []
human_ci_lower = []
human_ci_upper = []

pred_type = 'payoff'

for K_p in vary_K: 
    game = f'10.0*10.0*{K_p} pieces in a row wins.'
    samp_vals = []
    model_res = think_data['model_payoff'][model_src][game]
    
    for _ in range(n_bootstrap):
        sims = sum(model_res['sims'], [])
        boot_sims = np.random.choice(sims, k * n_sim_participants, replace=True)
        sim_group = analysis_utils.by_participants(list(boot_sims), n=n_sim_participants, m=k)
        model_judgments = list(analysis_utils.map_score_to_judgment(sim_group))
        m_scores_subsampled = [x['exp_util'] for x in model_judgments]
        samp_vals.append(np.mean(m_scores_subsampled))
    
    agg_vals.append(np.mean(samp_vals))
    model_ci_lower.append(np.percentile(samp_vals, 2.5))
    model_ci_upper.append(np.percentile(samp_vals, 97.5))
    
    h_vals = think_data['human_payoff'][game]
    vals = h_vals['payoff']
    samp_h_vals = []
    for _ in range(n_bootstrap): 
        samp_vals_h = np.random.choice(vals, n_sim_participants, replace=True)
        samp_h_vals.append(np.mean(samp_vals_h))
    h_agg_vals.append(np.mean(samp_h_vals))
    human_ci_lower.append(np.percentile(samp_h_vals, 2.5))
    human_ci_upper.append(np.percentile(samp_h_vals, 97.5))

agg_vals = np.array(agg_vals)
h_agg_vals = np.array(h_agg_vals)
model_ci_lower = np.array(model_ci_lower)
model_ci_upper = np.array(model_ci_upper)
human_ci_lower = np.array(human_ci_lower)
human_ci_upper = np.array(human_ci_upper)

# save payoff data to CSV (same format as funness)
payoff_df = pd.DataFrame({
    'game_id': [f'10.0*10.0*{K_p} pieces in a row wins.' for K_p in vary_K],
    'K': vary_K,
    'human_mean': h_agg_vals,
    'human_ci_lower': human_ci_lower,
    'human_ci_upper': human_ci_upper,
    'model_mean': agg_vals,
    'model_ci_lower': model_ci_lower,
    'model_ci_upper': model_ci_upper
})
payoff_df.to_csv(fig_data_file_pth + 'K_variation_payoff_predictions.csv', index=False)

model_yerr = [agg_vals - model_ci_lower, model_ci_upper - agg_vals]
human_yerr = [h_agg_vals - human_ci_lower, human_ci_upper - h_agg_vals]

line1 = ax.errorbar(vary_K, agg_vals, yerr=model_yerr, fmt='o', color='red', 
                    label='Intuitive Gamer', capsize=4, capthick=2, markersize=8)
line2 = ax.errorbar(vary_K, h_agg_vals, yerr=human_yerr, fmt='o', color='#34495E', 
                    label='Human', capsize=4, capthick=2, markersize=8)

ax.set_xlabel("K in a Row Wins", fontsize=20)
ax.set_ylabel("Payoff", fontsize=20)
ax.set_xticks(vary_K)
ax.tick_params(axis='both', labelsize=14)

ax = axes[1]

# Extract data from the CSV
K_vals = funness_df['K'].values
human_mean = funness_df['human_mean'].values
human_ci_low = funness_df['human_ci_lower'].values
human_ci_high = funness_df['human_ci_upper'].values
model_mean = funness_df['model_mean'].values
model_ci_low = funness_df['model_ci_lower'].values
model_ci_high = funness_df['model_ci_upper'].values

# Compute error bar arrays
model_yerr_fun = [model_mean - model_ci_low, model_ci_high - model_mean]
human_yerr_fun = [human_mean - human_ci_low, human_ci_high - human_mean]

line1 = ax.errorbar(K_vals, model_mean, yerr=model_yerr_fun, fmt='o', color='red', 
                    label='Intuitive Gamer', capsize=4, capthick=2, markersize=8)
line2 = ax.errorbar(K_vals, human_mean, yerr=human_yerr_fun, fmt='o', color='#34495E', 
                    label='Human', capsize=4, capthick=2, markersize=8)

ax.set_xlabel("K in a Row Wins", fontsize=20)
ax.set_ylabel("Funness", fontsize=20)
ax.set_xticks(vary_K)
ax.tick_params(axis='both', labelsize=14)
ax.set_ylim(0, 100)


fig.legend([line1, line2], ['Intuitive Gamer', 'Human'], 
           loc='lower center', ncol=2, fontsize=16, bbox_to_anchor=(0.5, -0.1))



plt.tight_layout()
plt.subplots_adjust(bottom=0.25) 
plt.savefig("payoff_and_funness_vs_K.jpg", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
'''
Make agg table
'''
importlib.reload(analysis_utils)
game_stats = []
for game, vals in think_data['human_fun'].items(): 
    board_rows, board_cols, win_conds = game.split("*")
    agg_fun = np.mean(vals)

    if 'Inf' in game or 'inf' in game: 
        board_fmt = f'InfxInf'
    else: 
        n_rows, n_cols, win_conds = game.split("*")
        n_rows = int(float(n_rows))
        n_cols = int(float(n_cols))
    
        board_fmt = f'{n_rows}x{n_cols}'

    payoff = np.mean(think_data['human_payoff'][game]['payoff'])
    pdraw = np.mean(think_data['human_payoff'][game]['pdraw'])
    pwin = np.mean(think_data['human_payoff'][game]['pwin'])
    game_stats.append([board_fmt, win_conds, agg_fun, payoff, pdraw, pwin])

game_stats = sorted(game_stats, key=lambda x: x[2], reverse=True)
analysis_utils.gen_latex_table(game_stats)


In [ ]:

# load in the temp zero file
temp0_file = "../model-data/think-exp/heuristics_temp0.txt"
temp0res = {}

with open(temp0_file, 'r') as f:
    contents = f.read()
results = eval(contents)

model_tag = 'temp0-ours'
think_data['model_payoff'][model_tag]  = {}
game_sims = {} # key = game id (board_rows*board_cols*win_condition), value are all simulations
for entry in results: 
    idx = entry["index"]
    game_id = idx2game[entry["index"]]
    game_sims[game_id] = entry  
    pdraw = entry['draw_percent']
    pwin = entry['win_percent']
    s = entry['game_scores']
    
    payoff = analysis_utils.compute_utility(pdraw, pwin)    
    payoff_obj = {'payoff': payoff, 'sims': s}
    temp0res[game_id] = {'payoff': payoff, 'sims': s}
    think_data['model_payoff'][model_tag][game_id] = payoff_obj


In [ ]:
'''
Scatterplots with k subsampling for model scores
'''
n_bootstrap = 1000
k = 6  
n_sim_participants =20
np.random.seed(7)
main_scatterplot_models = ['ours_full', 'expert', 'expert_mcts', 'random_terminal', 'temp0-ours', 'ours', 'depth3']
human_score_data = think_data['human_payoff']
ordered_games = sorted(list(human_score_data.keys()))
saved_r2 = {}


# sample the human res upfront 
human_bootstrap_means = [[] for _ in range(len(ordered_games))]
for bootstrap_idx in range(n_bootstrap):
    
    sampled_games = ordered_games
    sampled_game_indices = {} 
    
    for game_idx, game in enumerate(sampled_games):
        h_scores = human_score_data[game]['payoff']
        n_humans = len(h_scores)
        
        orig_idx = ordered_games.index(game)
        if orig_idx not in sampled_game_indices:
            sampled_game_indices[orig_idx] = []
        sampled_game_indices[orig_idx].append(game_idx)
        
        h_scores_resampled = np.random.choice(h_scores, n_humans, replace=True)
        h_mean = np.mean(h_scores_resampled)    
        human_bootstrap_means[orig_idx].append(h_mean)
human_ci_low = []
human_ci_high = []
human_mean_scores = [np.mean(human_bootstrap_means[game_idx]) for game_idx in range(len(ordered_games))]
for game_idx in range(len(ordered_games)):
        h_scores = human_bootstrap_means[game_idx]
        low, high = analysis_utils.compute_ci(h_scores, alpha=0.05)
        human_ci_low.append(max(np.mean(h_scores) - low, 0))
        human_ci_high.append(max(high - np.mean(h_scores), 0))


save_viz_data = {'means': [human_mean_scores], 'ci_low': [human_ci_low], 'ci_high': [human_ci_high], 'source': ['human']}

for model in main_scatterplot_models:
    np.random.seed(7)
    model_score_data = think_data['model_payoff'][model]
    skip_game_expert = "inf*inf*10 pieces in a row wins." # never finished/saved for expert -- filter out
    if model == 'expert':
        model_scores = [model_score_data[game]['payoff'] for game in ordered_games if game != skip_game_expert]
        human_scores = [human_score_data[game]['payoff'] for game in ordered_games if game != skip_game_expert]
    else: 
        model_scores = [model_score_data[game]['payoff'] for game in ordered_games]
        human_scores = [human_score_data[game]['payoff'] for game in ordered_games]
    human_mean_scores = [np.mean(s) for s in human_scores]
    model_mean_scores = [np.mean(s) for s in model_scores]
    
    fig, ax = plt.subplots()
    
    r2_samples = []
    
    model_ci_low = []
    model_ci_high = []

    if model == 'expert':
        sampled_games = [game for game in ordered_games if game != skip_game_expert]
    else: 

        sampled_games = ordered_games

    # map each kept game back to its position in the full ordered_games list,
    # then build human CI arrays aligned to sampled_games
    orig_indices = [ordered_games.index(g) for g in sampled_games]
    human_ci_low_m  = [human_ci_low[i]  for i in orig_indices]
    human_ci_high_m = [human_ci_high[i] for i in orig_indices]
    
    model_bootstrap_means = [[] for _ in range(len(sampled_games))]
    for bootstrap_idx in range(n_bootstrap):
        m_means = []
        h_means = []
        
        
        sampled_game_indices = {} 
        
        for game_idx, game in enumerate(sampled_games):
            h_scores = human_score_data[game]['payoff']
            m_scores = model_score_data[game]['payoff']
            n_humans = len(h_scores)
            
            true_idx = ordered_games.index(game)   # original index, for human lookup
            
            if true_idx not in sampled_game_indices:
                sampled_game_indices[true_idx] = []
            sampled_game_indices[true_idx].append(game_idx)
            
            if 'deepseek' in model or 'gpt' in model or 'llama' in model: # llm specific processing
                m_scores_subsampled = model_score_data[game]['payoff']
                # apply the same way of human subsampling on n_humans = 20
                m_scores_subsampled = np.random.choice(m_scores_subsampled, n_sim_participants, replace=True)
            else: 

            
                model_res = think_data['model_payoff'][model][game]
                try: 
                    sims = sum(model_res['sims'], [])
                except: 
                    sims = model_res['sims']

                boot_sims = np.random.choice(sims, k * n_sim_participants, replace=True)
                sim_group = analysis_utils.by_participants(list(boot_sims), n=n_sim_participants, m=k)
                model_judgments = list(analysis_utils.map_score_to_judgment(sim_group))
                m_scores_subsampled = [x['exp_util'] for x in model_judgments] #np.random.choice(model_judgments, k, replace=True)

            
            subsampled_mean = np.mean(m_scores_subsampled)
            m_means.append(subsampled_mean)
            h_means.append(human_bootstrap_means[true_idx][bootstrap_idx]) # reextract (true index)
            
            model_bootstrap_means[game_idx].append(subsampled_mean)  # position, not true index
        
        r2 = stats.pearsonr(h_means, m_means)[0] ** 2
        r2_samples.append(r2)
    
    
    model_mean_scores = [np.mean(model_bootstrap_means[game_idx]) for game_idx in range(len(sampled_games))]
    
    for game_idx in range(len(sampled_games)):
        game_bootstrap_means = model_bootstrap_means[game_idx]
        low, high = analysis_utils.compute_ci(game_bootstrap_means, alpha=0.05)

            
        model_ci_low.append(max(model_mean_scores[game_idx] - low, 0)) # need to ensure non-negative
        model_ci_high.append(max(high - model_mean_scores[game_idx], 0))
        
    
        
    full_r, pval  = stats.pearsonr([np.mean(model_bootstrap_means[game_idx]) for game_idx in range(len(sampled_games))], human_mean_scores)

    r2 = np.mean(r2_samples)
    r2_lower, r2_upper = analysis_utils.compute_ci(r2_samples)
    
    try:
        model_color = agent2color[model]
    except: model_color = 'purple'
    
    ax.set_title(f"R$^{{2}}$ = {round(r2, 2)} ({round(r2_lower, 2)}, {round(r2_upper, 2)})", fontsize=36)

    
    save_viz_data['means'].append(model_mean_scores)
    save_viz_data['ci_low'].append(model_ci_low)
    save_viz_data['ci_high'].append(model_ci_high)
    save_viz_data['source'].append(model)

    sns.regplot(x=model_mean_scores, y=human_mean_scores, ax=ax, truncate=False, ci=95, 
                color = model_color)

    saved_r2[model] = r2_samples
    
    ax.errorbar(
        x=model_mean_scores, 
        y=human_mean_scores,
        yerr=[human_ci_low_m, human_ci_high_m],  
        xerr=[model_ci_low, model_ci_high],  
        alpha=0.3,
        fmt='none', 
        capsize=5, 
        zorder=1, 
        color = model_color
    )
    
    lb = -1.1
    ub = 1.1
    ref = np.linspace(lb, ub, 100)
    ax.plot(ref, ref, '--', color='red', alpha=0.3)
    ax.set_ylim([lb, ub])
    ax.set_xlim([lb, ub])
    ax.set_ylabel("Human", fontsize=30)
    ax.set_xlabel("Model", fontsize=30)
    ax.grid(False)
    
    plt.tight_layout()
    plt.savefig(f"{main_save}/think/{model}_agg_payoff.svg", dpi=400,bbox_inches='tight')
    plt.close()

save_viz_data = pd.DataFrame(save_viz_data)
save_viz_data.to_csv(fig_data_file_pth + 'scatterplot_payoffs.csv')


# Persist saved_r2 so param_ablations can read it without re-bootstrapping.
import os
os.makedirs(f"{main_dir}think", exist_ok=True)
with open(f"{main_dir}think/saved_r2.json", "w") as _f:
    json.dump({m: list(map(float, vs)) for m, vs in saved_r2.items()}, _f)

In [ ]:
len(ordered_games)

In [ ]:
orig_idx

In [ ]:
agent2color['opt'] = 'black'

In [ ]:

n_bootstrap = 1000
k = 6
n_sim_participants = 20
eps = 0.01
np.random.seed(7)
ordered_games = sorted(list(human_score_data.keys()))
converged_games = {}
for game in ordered_games: 
    sim_pred = think_data['model_payoff']['expert_mcts'][game]['payoff']
    m = np.mean(sim_pred)
    if not (analysis_utils.epislon_equiv(m, 0, eps) or analysis_utils.epislon_equiv(m, -1, eps) or analysis_utils.epislon_equiv(m, 1, eps)): 
        game_type = game2game_type[game]
        if not ((game_type == 'simple-win') or (game_type == 'rectangle-board')): 
            continue
        n_rows, n_cols, win_conds = game.split("*")
        n_rows = int(float(n_rows))
        n_cols = int(float(n_cols))
        k_win = int(win_conds.split(' pieces in a row')[0])
        res = analysis_utils.opt_game_outcome_mnk(n_rows, n_cols, k_win)
        if res is not None: 
            converged_games[game] = res
    else: 
        converged_games[game] = m

human_score_data = think_data['human_payoff']
ordered_convg_games = sorted(list(converged_games.keys()))

for source in ['human'] + list(think_data['model_payoff'].keys()):
    if source not in model2name: continue
    if source == 'human':
        model_score_data = None
        comp_scores = [human_score_data[game]['payoff'] for game in ordered_convg_games]
    else:
        model_score_data = think_data['model_payoff'][source]
        comp_scores = []
        for game in ordered_convg_games:
            if 'deepseek' in source or 'gpt' in source or 'llama' in source:
                # r2_lower, r2_upper = analysis_utils.compute_ci(r2_samples)
                sims = model_score_data[game]['payoff']
                boot_sims = np.random.choice(sims, n_sim_participants, replace=True) 
                comp_scores.append(boot_sims)
            else:
                model_res = model_score_data[game]
                try: 
                    sims = sum(model_res['sims'], [])
                except: 
                    sims = model_res['sims']
                boot_sims = list(np.random.choice(sims, k * n_sim_participants, replace=True))
                sim_group = analysis_utils.by_participants(boot_sims, n=n_sim_participants, m=k)
                judgments = list(analysis_utils.map_score_to_judgment(sim_group))
                comp_scores.append([j['exp_util'] for j in judgments])
    
    opt_scores = [converged_games[game] for game in ordered_convg_games]
    source_mean = [np.mean(s) for s in comp_scores]
    
    source_preds = []
    for s in source_mean: 
        if s > 0.5: source_preds.append(1)
        elif s < -0.5: source_preds.append(-1)
        else: source_preds.append(0)
    acc = np.sum(np.array(source_preds) == np.array(opt_scores)) / len(opt_scores)
    print(model2name[source], round(acc,2)*100, "%")

    bootstrap_means = [[] for _ in range(len(ordered_convg_games))]
    for _ in range(n_bootstrap):
        for idx, game in enumerate(ordered_convg_games):
            scores = comp_scores[idx]
            resample = np.random.choice(scores, len(scores), replace=True)
            bootstrap_means[idx].append(np.mean(resample))
    
    ci_low = []
    ci_high = []
    for idx, means in enumerate(bootstrap_means):
        low, high = analysis_utils.compute_ci(means, alpha=0.05)
        ci_low.append(np.max(source_mean[idx] - low, 0))
        ci_high.append(np.max(high - source_mean[idx], 0))
    

    if source == 'human':
        r2_samples = []
        for _ in range(n_bootstrap): 
            h_means = []
            m_means = []
            for game in ordered_convg_games: 
                h_scores = human_score_data[game]['payoff']
                m_score = converged_games[game]
                h_resample = np.random.choice(h_scores, len(h_scores), replace=True)
                h_means.append(np.mean(h_resample))
                m_means.append(m_score)
            r2 = stats.pearsonr(h_means, m_means)[0] ** 2
            r2_samples.append(r2)
        r2 = np.mean(r2_samples)
        r2_lower, r2_upper = analysis_utils.compute_ci(r2_samples)
    else: 
        r2 = stats.pearsonr(source_mean, opt_scores)[0] ** 2
        r_low = r2
        r_high = r2

    fig, ax = plt.subplots()
    
    if source == 'human':
        color = agent2color['opt']#'black'
        color = "#A93226" 
        x_lab = 'Model'

        # save out
        save_viz_data = {'means': [source_mean, opt_scores],
                          'ci_low': [[max(v,0) for v in ci_low], []], 
                          'ci_high': [[max(v,0) for v in ci_high], []],
                         'source': ['human', 'gt_opt']}
        save_viz_data = pd.DataFrame(save_viz_data)
        save_viz_data.to_csv(fig_data_file_pth + 'gt_opt_scatter.csv')


    else:
        try: 
            color = agent2color[source]
        except: color = 'purple'
        x_lab = "Game-Theoretic Optimal"
    ax.set_title(f"R$^{{2}}$ = {round(r2, 2)} ({round(r2_lower, 2)}, {round(r2_upper, 2)})", fontsize=28)
    sns.regplot(x=opt_scores, y=source_mean, ax=ax, truncate=False, ci=95, color=color)#"grey")
    ax.errorbar(
        x=opt_scores,
        y=source_mean,
        yerr=[[max(v,0) for v in ci_low], [max(v,0) for v in ci_high]],
        alpha=0.3,
        fmt='none',
        capsize=5,
        zorder=1,
        color=color
    )
    
    lb = -1.1
    ub = 1.1
    ref = np.linspace(lb, ub, 100)
    ax.plot(ref, ref, '--', color='red', alpha=0.3)
    ax.set_ylim([lb, ub])
    ax.set_xlim([lb, ub])
    ax.grid(False)
    ax.set_ylabel(f"{model2name[source]}", fontsize=26)
    ax.set_xlabel(x_lab, fontsize=26)
    


    plt.tight_layout()
    plt.savefig(f"{main_save}/think/opt/{source}_opt_payoff.svg", dpi=400)
    plt.close()


sources = ['human',
 'ours_full',
 'depth3',
 'random_terminal',
 'expert',
 'expert_mcts',

 'temp0-ours'
 ]
games = sorted(converged_games.keys())

# Function to get binary correctness vector per source
# +1 => correct (match optimal), -1 => incorrect, 0 => neutral (tie or no strong prediction)
# help from claude
def get_correctness(source):
    # mean payoff per game
    means = []
    for game in games:
        if source == 'human':
            sims = think_data['human_payoff'][game]['payoff']
            means.append(np.mean(sims))
        else:
            sims = think_data['model_payoff'][source][game]['payoff']
            print(sims)
            try: 
                means.append(np.mean(np.random.choice(sims, n_sim_participants, replace=True)))
            except: 
                means.append(np.mean(sims))
    # binary predictions
    preds = np.zeros(len(means), dtype=int)
    preds[np.array(means) > 0.5] = 1
    preds[np.array(means) < -0.5] = -1
    # optimal outcomes
    opts = np.array([converged_games[game] for game in games], dtype=int)
    # correctness: 1 if matches, else 0
    return (preds == opts).astype(int)


df = pd.DataFrame({src: get_correctness(src) for src in sources}, index=games)

df_accuracy = df.mean().rename('accuracy') * 100



In [ ]:
from scipy.stats import pearsonr 
def get_bootstrapped_scores(k, task, model, n_sim_participants=20, debug_print=False, pseudodraw_count=0):
    # get bootstrapped decomposed scores
    human_scores = []
    model_scores = []
    
    human_var = []
    model_var= [ ]

    for game_id in ordered_games:
        game_df = human_df[(human_df.game_id == game_id) & (human_df.task == 'advantage')].reset_index()
        if game_df.empty:
            continue

        entry = game_df.iloc[0]
        scores = []
        for score_entry in eval(entry.all_scores):
            draw = score_entry["draw_response"] / 100
            p1 = score_entry["firstplayer_response"] / 100
            pwin = p1 * (1 - draw)
            
            utility = analysis_utils.compute_utility(draw, pwin)

            if task == "advantage-draw":
                score = draw
            elif task == "advantage-win":
                score = pwin
            elif task == "advantage-p1nodraw":
                score = p1
            elif task == "advantage-utility":
                score = utility
            scores.append(score)

        if not scores:
            continue
        
        if model == 'split-human': 
            # run split-halves, otherwise run with everything
            random.shuffle(scores)
            s1, s2 = np.array_split(scores, 2)
            if debug_print: 
                print(len(s1), len(s2), np.mean(s1), np.mean(s2))
                print(s1, s2)
            human_scores.append(np.mean(s1))
            human_var.append(np.std(s1))
            model_scores.append(np.mean(s2))
            model_var.append(np.std(s2))
        else: 
        
            boot_human = np.random.choice(scores, n_sim_participants, replace=True)
            human_scores.append(np.mean(boot_human)) 
            human_var.append(np.std(scores))

            model_res = think_data['model_payoff'][model][game_id]
            sims = sum(model_res['sims'], [])

            if pseudodraw_count != 0: 
                pseudodraw_count_list = [0 for _ in range(pseudodraw_count)]
                sims = list(sims) + list(pseudodraw_count_list) 

            boot_sims = np.random.choice(sims, k * n_sim_participants, replace=True)
            sim_group = analysis_utils.by_participants(list(boot_sims), n=n_sim_participants, m=k)
            
            model_judgments = list(analysis_utils.map_score_to_judgment(sim_group))
            
            pwin_list = [g.count(1) / len(g) for g in sim_group]

            p1_list = [g.count(1) / (len(g) - g.count(0))
                       if g.count(0) != len(g) else 0.5 
                       for g in sim_group]
            pdraw_list = [g.count(0) / len(g) for g in sim_group]
            if debug_print: 
                print(game_id)
                print(p1_list)
                print([g.count(0) for g in sim_group])
                print(sim_group)
            payoff_list = [x['exp_util'] for x in model_judgments]

            sim_scores = []
            for i in range(n_sim_participants):
                if task == "advantage-draw":
                    score = pdraw_list[i]
                elif task == "advantage-win":
                    score = pwin_list[i]
                elif task == "advantage-p1nodraw": 
                    score = p1_list[i]
                    
                elif task == "advantage-utility":
                    score = payoff_list[i]
                sim_scores.append(score)

            model_scores.append(np.mean(sim_scores))
            model_var.append(np.std(sim_scores)) 

    return human_scores, model_scores, human_var, model_var

In [ ]:
# get split-halve for humans
tasks = [ "advantage-win",
         "advantage-p1nodraw", 
         "advantage-draw", 
        "advantage-utility"
         ]
num_bootstrap= 1000
for task in tasks: 
    r2s = []
    for _ in range(num_bootstrap):
        debug_print = False
        hs, ms, _, _ = get_bootstrapped_scores(k, task, 'split-human', 
                                               n_sim_participants, debug_print=debug_print)
        
        r2 = pearsonr(hs, ms)[0] **2
        r2s.append(r2)
        
    print(task, np.mean(r2s), np.percentile(r2s, 2.5), np.percentile(r2s, 97.5))

In [ ]:
def plot_human_vs_model_scatter_by_task(
    comp_models,
    color_for_model,
    name_for_model,
    tasks,
    k=6,
    n_sim_participants=20,
    num_bootstrap=1000,
    save_dir="final_figures/think",
    seed=7,
):
    """Per-task human-vs-model scatter with bootstrap CI error bars.

    Args:
        comp_models: list of model keys to draw, one panel per model.
        color_for_model: callable model_key -> matplotlib color used for both
            scatter and errorbars.
        name_for_model:  callable model_key -> display name used in the title.
        tasks:           list of advantage-* task strings; one figure per task.
    """
    os.makedirs(save_dir, exist_ok=True)
    np.random.seed(seed)
    game_means = {}

    for task in tasks:
        fig, axes = plt.subplots(
            1, len(comp_models),
            figsize=(5 * len(comp_models), 8),
            sharex=True, sharey=True,
        )
        if len(comp_models) == 1:
            axes = [axes]

        for model_idx, model in enumerate(comp_models):
            ax = axes[model_idx]
            all_human_scores, all_model_scores = [], []
            r2s = []
            for _ in range(num_bootstrap):
                hs, ms, _, _ = get_bootstrapped_scores(
                    k, task, model, n_sim_participants, debug_print=False
                )
                all_human_scores.append(hs)
                all_model_scores.append(ms)
                r2s.append(pearsonr(hs, ms)[0] ** 2)
            all_human_scores = np.array(all_human_scores)
            all_model_scores = np.array(all_model_scores)
            mean_human = np.mean(all_human_scores, axis=0)
            mean_model = np.mean(all_model_scores, axis=0)

            # asymmetric CIs per game
            num_games = len(mean_human)
            yerr_low, yerr_high, xerr_low, xerr_high = [], [], [], []
            for g in range(num_games):
                h_lo, h_hi = analysis_utils.compute_ci(all_human_scores[:, g])
                m_lo, m_hi = analysis_utils.compute_ci(all_model_scores[:, g])
                yerr_low.append(max(mean_human[g] - h_lo, 0))
                yerr_high.append(max(h_hi - mean_human[g], 0))
                xerr_low.append(max(mean_model[g] - m_lo, 0))
                xerr_high.append(max(m_hi - mean_model[g], 0))

            color = color_for_model(model)
            ax.scatter(mean_model, mean_human, alpha=0.6, c=color)
            for i in range(num_games):
                ax.errorbar(
                    mean_model[i], mean_human[i],
                    xerr=[[xerr_low[i]], [xerr_high[i]]],
                    yerr=[[yerr_low[i]], [yerr_high[i]]],
                    fmt='none', alpha=0.4, c=color,
                )

            for idx, m_val in enumerate(mean_model):
                game_means[f"{model}_k={k}_game{idx}"] = m_val

            try:
                r2 = float(np.mean(r2s))
                r2_l = float(np.percentile(r2s, 2.5))
                r2_u = float(np.percentile(r2s, 97.5))
                print(model, r2)
                ax.set_title(
                    f"{name_for_model(model)}\n$R^2$={r2:.2f} ({r2_l:.2f}, {r2_u:.2f})",
                    fontsize=26,
                )
            except Exception:
                ax.set_title(f"{name_for_model(model)}\n$R^2$=nan", fontsize=20)

            lb, ub = (-1.05, 1.05) if 'utility' in task else (-0.05, 1.05)
            ax.set_xlim(lb, ub)
            ax.set_ylim(lb, ub)
            ax.plot([lb, ub], [lb, ub], 'r--', alpha=0.7)
            if model_idx == 0:
                ax.set_ylabel("Human", fontsize=24)
            ax.set_xlabel("Model", fontsize=24)
            ax.set_aspect('equal')

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(f"{save_dir}/{task}.pdf", dpi=400)
        plt.show()

    return game_means


In [ ]:
game_means = plot_human_vs_model_scatter_by_task(
    comp_models=['ours_full', 'random_terminal', 'expert', 'expert_mcts'],
    color_for_model=lambda m: agent2color[m],
    name_for_model=lambda m: model2name[m],
    tasks=["advantage-win", "advantage-p1nodraw", "advantage-draw", "advantage-utility"],
)


In [ ]:
model2name_partial = {'ours': 'IGM (Partial Sims)', 'ours_full': 'IGM (Full Sims)'}
game_means = plot_human_vs_model_scatter_by_task(
    comp_models=['ours', 'ours_full'],
    color_for_model=lambda m: 'red' if 'full' in m else 'maroon',
    name_for_model=lambda m: model2name_partial[m],
    tasks=["advantage-win", "advantage-p1nodraw", "advantage-draw", "advantage-utility"],
)


In [ ]:
''' 
Get split-half corrs
'''
n_split_corrs = 1000

resp_versions = ['think', 'play', 'watch']
split_corrs_per_resp = {resp_version: [ ] for resp_version in resp_versions}

np.random.seed(7)
random.seed(7)

human_score_data = think_data['human_payoff']
ordered_games = sorted(list(human_score_data.keys()))
print(len(ordered_games))
human_scores = [human_score_data[game]['payoff'] for game in ordered_games]
human_mean_scores = [np.mean(s) for s in human_scores]

r2_samples = []


for _ in range(n_bootstrap): 
        
    human_s1 = []
    human_s2 = []
    for game in ordered_games: 
        h_scores = human_score_data[game]['payoff']
        random.shuffle(h_scores)
        s1, s2 = np.array_split(h_scores, 2)
        human_s1.append(np.mean(s1))
        human_s2.append(np.mean(s2))
    
        
        
    r2 = stats.pearsonr(human_s1, human_s2)[0] ** 2
    r2_samples.append(r2)

r2 = np.mean(r2_samples)
r2_lower, r2_upper = analysis_utils.compute_ci(r2_samples)

r2, r2_lower, r2_upper

In [ ]:

sources = ['human', 
           'ours_full', 
           'depth3', 'expert', 'expert_mcts', 

           'random_terminal',
           ]
model2name['human'] = 'Human'
games = sorted(converged_games.keys())
n_boot = 1000 

ordered_convg_games = sorted(converged_games.keys())
n_games = len(ordered_convg_games)
n_sources = len(sources)
n_sim_participants =20
k=6

def get_boot_payoffs(source):
    boot_scores = []
    for game in ordered_convg_games:
        if source == 'human':
            sims = human_score_data[game]['payoff']
            boot_sims = np.random.choice(sims, n_sim_participants, replace=True)
        else:
            model_score_data = think_data['model_payoff'][source]
            if 'deepseek' in source or 'gpt' in source or 'llama' in source:
                sims = model_score_data[game]['payoff']
                boot_sims = np.random.choice(sims, n_sim_participants, replace=True)
            else:
                model_res = model_score_data[game]
                sims = sum(model_res['sims'], [])
                boot_sims = list(np.random.choice(sims, k * n_sim_participants, replace=True))
                sim_group = analysis_utils.by_participants(boot_sims, n=n_sim_participants, m=k)
                judgments = list(analysis_utils.map_score_to_judgment(sim_group))
                boot_sims = [j['exp_util'] for j in judgments]
        boot_scores.append(np.mean(boot_sims))  # mean for each game
    return np.array(boot_scores)

# pre-compute optimal payoffs (non-bootstrapped -- deterministic)
optimal_payoffs = np.array([converged_games[game] for game in ordered_convg_games])

r2_matrices = np.zeros((n_boot, n_sources, n_sources))


all_source_diffs = []

for boot in tqdm(range(n_boot)):
    # for each source, compute a vector of mean bootstrapped payoffs for each game
    source_diffs = []
    for source in sources:
        payoffs = get_boot_payoffs(source)
        diff_from_opt = payoffs - optimal_payoffs
        source_diffs.append(diff_from_opt)
        
        all_source_diffs.append([source, diff_from_opt])
        
        
    # shape: (n_sources, n_games)
    diffs_matrix = np.vstack(source_diffs)

    for i in range(n_sources):
        for j in range(n_sources):
            if np.std(diffs_matrix[i]) == 0 or np.std(diffs_matrix[j]) == 0:
                r2 = np.nan  
            else:
                r = np.corrcoef(diffs_matrix[i], diffs_matrix[j])[0, 1]
                r2 = r ** 2
            r2_matrices[boot, i, j] = r2

mean_r2 = np.nanmean(r2_matrices, axis=0)
lower_r2 = np.nanpercentile(r2_matrices, 2.5, axis=0)
upper_r2 = np.nanpercentile(r2_matrices, 97.5, axis=0)

df_mean = pd.DataFrame(mean_r2, index=sources, columns=sources)
df_lower = pd.DataFrame(lower_r2, index=sources, columns=sources)
df_upper = pd.DataFrame(upper_r2, index=sources, columns=sources)

for i, src1 in enumerate(sources):
    for j, src2 in enumerate(sources):
        print(f"{src1} vs {src2}: mean R^2={df_mean.iloc[i, j]:.2f} ({df_lower.iloc[i, j]:.2f}–{df_upper.iloc[i, j]:.2f})")




In [ ]:
latex_rows = []

win_thresh= 0.5

for source in sources:
    if source == 'ours' or source == 'random': continue # these were the partial game sim cases
    if source == 'human':
        model_score_data = None
        comp_scores = [human_score_data[game]['payoff'] for game in ordered_convg_games]
    else:
        model_score_data = think_data['model_payoff'][source]
        comp_scores = []
        for game in ordered_convg_games:
            if 'deepseek' in source or 'gpt' in source or 'llama' in source:
                if 'r1' in source:
                    print(source) 
                    print('game: ', game, np.median(model_score_data[game]['payoff']), converged_games[game])
                comp_scores.append(model_score_data[game]['payoff'])
            else:
                model_res = model_score_data[game]
                try: 
                    sims = sum(model_res['sims'], [])
                except: sims = model_res['sims']
                boot_sims = list(np.random.choice(sims, k * n_sim_participants, replace=True))
                sim_group = analysis_utils.by_participants(boot_sims, n=n_sim_participants, m=k)
                judgments = list(analysis_utils.map_score_to_judgment(sim_group))
                comp_scores.append([j['exp_util'] for j in judgments])
    
    opt_scores = [converged_games[game] for game in ordered_convg_games]
    source_mean = [np.mean(s) for s in comp_scores]


    acc_samples = []
    
    dev_samples = []
    
    r2_samples = []
    
    for _ in range(n_bootstrap):
        sample_means = []
        for idx in range(len(comp_scores)):
            resample = np.random.choice(comp_scores[idx], len(comp_scores[idx]), replace=True)
            sample_means.append(np.mean(resample))
        preds = [1 if s > win_thresh else -1 if s < -win_thresh else 0 for s in sample_means]
        acc_samples.append(np.sum(np.array(preds) == np.array(opt_scores)) / len(opt_scores))
        
        devs= []
        for pred, opt_val in zip(sample_means,opt_scores): 
            dist = np.abs(opt_val - pred)
            devs.append(dist)
        dev_samples.append(np.mean(devs))
        
        r2_samples.append(pearsonr(sample_means,opt_scores)[0] **2) 
        
    acc_ci_low, acc_ci_high = analysis_utils.compute_ci(acc_samples)
    dev_ci_low, dev_ci_high = analysis_utils.compute_ci(dev_samples)
    acc = np.mean(acc_samples)
    dev = np.mean(dev_samples)
    
    r2 = np.mean(r2_samples)
    r2_ci_low, r2_ci_high = analysis_utils.compute_ci(r2_samples)

    model_name = model2name[source]
    row = f"{model_name} & {acc:.2f} ({acc_ci_low:.2f}, {acc_ci_high:.2f}) & {r2:.2f} ({r2_ci_low:.2f}, {r2_ci_high:.2f}) & {dev:.2f} ({dev_ci_low:.2f}, {dev_ci_high:.2f}) \\\\"
    latex_rows.append(row)

print("\\begin{table}[h!]")
print("\\centering")
print("\\begin{tabular}{lccc}")
print("\\toprule")
print("\\textbf{Source} & \\textbf{Accuracy (95\\% CI)} & \\textbf{R$^2$ (95\\% CI)} & \\textbf{Deviation (95\\% CI)} \\\\")
print("\\midrule")
for row in latex_rows:
    print(row)
print("\\bottomrule")
print("\\end{tabular}")
print("\\caption{Model and human prediction accuracy and correlation (R$^2$) with game-theoretic optima. Bootstrap 95\\% confidence intervals are shown in brackets.}")
print("\\label{tab:model_accuracy_correlation}")
print("\\end{table}")


In [ ]:
''' 
Empirical payoffs vs pred payoff
'''
with open("final_processed_res/play/empirical_outcomes.json", "r") as f: 
    empirical_outcomes = json.load(f)
   
empirical_payoffs = {game: 0 for game in empirical_outcomes}
for game, outcomes in empirical_outcomes.items():
    outcomes = [outcome if outcome != 2 else -1 for outcome in outcomes] # relative to player 1 
    p_win = outcomes.count(1) / len(outcomes)
    p_draw = outcomes.count(0) / len(outcomes)
    print(game, p_draw, p_win, outcomes)
    payoff= analysis_utils.compute_utility(p_draw, p_win)
    empirical_payoffs[game] = payoff

In [ ]:
comp_scores_mean = [converged_games[game] for game in ordered_convg_games if game in empirical_payoffs]

emp_scores = [empirical_payoffs[game] for game in empirical_outcomes]

agents =['human', 
         'ours_full', 
         'expert', 'expert_mcts', 'random_terminal', 
         ]

ordered_games = [game for game in sorted(list(empirical_payoffs)) if game in converged_games]
ordered_games = sorted(list(empirical_payoffs)) 

emp_scores = [empirical_payoffs[game] for game in ordered_games]


n_boot = 1000
np.random.seed(7)
fig, axes = plt.subplots(2,3,figsize=(10,8))
title_size = 18
axes = axes.flatten()
agent2color['human'] = 'pink'
agent2color['split-human'] = 'pink'


save_viz_data = {'means': [emp_scores],
                        'ci_low': [[]], 
                        'ci_high': [[]],
                        'source': ['empirical_outcomes']}


for agent_idx, agent in enumerate(agents): 
        ax = axes[agent_idx + 1] # start with gt
        all_comp_scores_mean = np.zeros([n_boot, len(ordered_games)])
        r2_samples = []

        for boot_i in range(n_boot): 
                comp_scores = []
                if agent == 'human': 
                        n_sim_participants = 20 # real participants (defined for subsampling)
                        comp_scores = []
                        for game in ordered_games: 
                                resps = human_score_data[game]['payoff']

                                boot_resps = np.random.choice(resps, n_sim_participants, replace=True)
                                comp_scores.append(boot_resps)
                elif 'deepseek' in agent or 'gpt' in agent or 'llama' in agent: 
                        for game in ordered_games: 
                                resps = think_data['model_payoff'][agent][game]['payoff']
                                boot_resps = np.random.choice(resps, n_sim_participants, replace=True)
                                comp_scores.append(boot_resps)
                else: 
                        k = 6
                        n_sim_participants = 20
                        for game in ordered_games: 
                                model_res = think_data['model_payoff'][agent][game]
                                sims = sum(model_res['sims'], [])


                                boot_sims = np.random.choice(sims, k * n_sim_participants, replace=True)
                                sim_group = analysis_utils.by_participants(list(boot_sims), n=n_sim_participants, m=k)
                                model_judgments = list(analysis_utils.map_score_to_judgment(sim_group))
                                m_scores_subsampled = [x['exp_util'] for x in model_judgments] 
                                comp_scores.append(m_scores_subsampled)

                comp_scores_mean = [np.mean(s) for s in comp_scores]
                all_comp_scores_mean[boot_i, :] = comp_scores_mean
                r2 = stats.pearsonr(comp_scores_mean, emp_scores)[0]**2
                r2_samples.append(r2)

        r2 = np.mean(r2_samples)
        r2_lower = np.percentile(r2_samples, 2.5)
        r2_upper = np.percentile(r2_samples, 97.5)

        comp_scores_mean = all_comp_scores_mean.mean(axis=0)
        comp_scores_ci_lower = np.percentile(all_comp_scores_mean, 2.5, axis=0)
        comp_scores_ci_upper = np.percentile(all_comp_scores_mean, 97.5, axis=0)
        comp_scores_err_lower = comp_scores_mean - comp_scores_ci_lower
        comp_scores_err_upper = comp_scores_ci_upper - comp_scores_mean
        comp_scores_err_lower =[max(x,0) for x in comp_scores_err_lower]
        comp_scores_err_upper =[max(x,0) for x in comp_scores_err_upper]

        comp_scores_err = np.vstack([comp_scores_err_lower, comp_scores_err_upper])
        sns.regplot(x=comp_scores_mean, y=emp_scores, ax=ax, truncate=False, ci=95, 
                    color=agent2color[agent])
        
        save_viz_data['means'].append(comp_scores_mean)
        save_viz_data['ci_low'].append(comp_scores_err_lower)
        save_viz_data['ci_high'].append(comp_scores_err_upper)
        save_viz_data['source'].append(agent)
        ax.errorbar(x=comp_scores_mean, y=emp_scores,
                xerr=comp_scores_err,
                color=agent2color[agent],
                alpha=0.3,
                fmt='none', capsize=5, zorder=1)
        lb = -1.2
        ub = 1.2
        ref = np.linspace(lb, ub, 100)
        ax.plot(ref, ref, '--', color='red', alpha=0.3)

        ax.set_ylim([lb, ub])
        ax.set_xlim([lb, ub])
        ax.set_ylabel("Empirical Payoff", fontsize=18)
        if agent == 'human': 
                ax.set_xlabel("Human", fontsize=18)
        else: ax.set_xlabel("Model", fontsize=18)
        model_name = model2name[agent]
        if '(Full)' in model_name: 
                model_name = model_name.split(" (Full)")[0] # assume all full for now
        ax.set_aspect('equal', adjustable='box')  

        # Rest of your title code
        model_name = model2name[agent]
        if 'o3' in model_name or 'o1' in model_name: 
                ax.set_title(f"{model_name}\nR$^{{2}}$ = {round(r2, 2)}", fontsize=title_size)
        else: 
                ax.set_title(f"{model_name}\nR$^{{2}}$ = {round(r2, 2)} ({round(r2_lower, 2)}, {round(r2_upper, 2)})", fontsize=title_size)


comp_scores_mean = [converged_games[game] for game in ordered_convg_games if game in empirical_payoffs]

emp_scores = [empirical_payoffs[game] for game in ordered_convg_games if game in empirical_payoffs]

ax = axes[0]#-1]
r, pval, (r_lower, r_upper) = analysis_utils.pearsonr_ci(
                                                         emp_scores,
                                                         comp_scores_mean)

sns.regplot(x=comp_scores_mean, y=emp_scores, ax=ax, truncate=False, ci=95, color="grey")
ax.errorbar(x=comp_scores_mean, y=emp_scores,
        alpha=0.3,
        fmt='none', capsize=5, zorder=1, color="grey")

save_viz_data['means'].append(comp_scores_mean)
save_viz_data['ci_low'].append([])
save_viz_data['ci_high'].append([])
save_viz_data['source'].append('gt_opt')

save_viz_data['means'].append(emp_scores)
save_viz_data['ci_low'].append([])
save_viz_data['ci_high'].append([])
save_viz_data['source'].append('empirical_subset_gt_match')

lb = -1.2
ub = 1.2
ref = np.linspace(lb, ub, 100)
ax.plot(ref, ref, '--', color='red', alpha=0.3)
ax.set_ylim([lb, ub])
ax.set_xlim([lb, ub])
ax.set_ylabel("Empirical Payoff", fontsize=18)
ax.set_xlabel("Game-Theoretic", fontsize=18)
ax.set_aspect('equal', adjustable='box')
ax.set_title(f"Game-Theoretic Optimal\nR$^{2}$ = {round(r ** 2, 2)}", fontsize=title_size) 


plt.tight_layout(rect=[0, 0, 1, 1], pad=2.0)  # pad increases space between plots
plt.savefig("empirical_outcome_models_play.pdf", dpi=400)


save_viz_data = pd.DataFrame(save_viz_data)
save_viz_data.to_csv(fig_data_file_pth + 'empirical_outcomes.csv')